<a href="https://colab.research.google.com/github/mattshaabani/sql-finetuning-pipeline/blob/main/notebooks/02_training_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQL Generation Fine-tuning on Google Colab

Fine-tuning TinyLlama-1.1B for text-to-SQL using LoRA.

**Runtime required:** GPU (T4 or better)
Go to: Runtime → Change runtime type → T4 GPU

**What this notebook does:**
1. Clones the project from GitHub
2. Uses Colab's native package versions (no version conflicts)
3. Loads Spider dataset
4. Fine-tunes TinyLlama with LoRA
5. Evaluates base vs fine-tuned model
6. Pushes model to HuggingFace Hub

In [1]:
import subprocess

# Only install what Colab doesn't have
# Let Colab's pre-installed transformers/peft/torch stay as-is
packages = [
    "datasets",
    "mlflow",
    "nltk",
    "pydantic",
    "pydantic-settings",
    "pyyaml",
    "python-dotenv",
]

for pkg in packages:
    subprocess.run(["pip", "install", "-q", pkg])
    print(f"✓ {pkg}")

# Check what versions Colab has
import transformers, peft, torch
print(f"\nColab's pre-installed versions:")
print(f"  torch:        {torch.__version__}")
print(f"  transformers: {transformers.__version__}")
print(f"  peft:         {peft.__version__}")
print(f"  CUDA:         {torch.cuda.is_available()}")

✓ datasets
✓ mlflow
✓ nltk
✓ pydantic
✓ pydantic-settings
✓ pyyaml
✓ python-dotenv

Colab's pre-installed versions:
  torch:        2.11.0+cu128
  transformers: 5.10.2
  peft:         0.19.1
  CUDA:         True


In [4]:
# Verify peft and transformers work together
import torch
from transformers import AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM

print(f"torch:        {torch.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}")

# Quick compatibility test with a tiny model
print("\nTesting LoRA compatibility...")
try:
    tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
    print("✓ Tokenizer loads fine")
except Exception as e:
    print(f"✗ Tokenizer error: {e}")

try:
    config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    print("✓ LoraConfig created fine")
except Exception as e:
    print(f"✗ LoraConfig error: {e}")

print("\nCompatibility check done")

torch:        2.11.0+cu128
CUDA:         True

Testing LoRA compatibility...
✓ Tokenizer loads fine
✓ LoraConfig created fine

Compatibility check done


In [2]:
import os
import subprocess

if not os.path.exists("sql-finetuning-pipeline"):
    subprocess.run([
        "git", "clone",
        "https://github.com/MattShaabani/sql-finetuning-pipeline.git"
    ])

os.chdir("sql-finetuning-pipeline")
subprocess.run(["pip", "install", "-q", "-e", "."])

print(f"Working directory: {os.getcwd()}")
print("Project installed")

Working directory: /content/sql-finetuning-pipeline
Project installed


In [3]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Make sure you selected T4 GPU runtime.")

Thu Jun 18 14:56:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
import os
from getpass import getpass

os.environ["HF_TOKEN"]          = getpass("Enter HuggingFace token: ")
os.environ["HF_USERNAME"]       = "mattiinn"
os.environ["LOG_LEVEL"]         = "INFO"
os.environ["APP_ENV"]           = "production"
os.environ["ANTHROPIC_API_KEY"] = ""
os.environ["CHROMA_HOST"]       = "localhost"
os.environ["CHROMA_PORT"]       = "8000"
os.environ["QDRANT_HOST"]       = "localhost"
os.environ["QDRANT_PORT"]       = "6333"
os.environ["OLLAMA_BASE_URL"]   = "http://localhost:11434"
os.environ["OLLAMA_MODEL"]      = "llama3.1"

with open(".env", "w") as f:
    for key in ["HF_TOKEN", "HF_USERNAME", "LOG_LEVEL",
                "APP_ENV", "ANTHROPIC_API_KEY", "CHROMA_HOST",
                "CHROMA_PORT", "QDRANT_HOST", "QDRANT_PORT",
                "OLLAMA_BASE_URL", "OLLAMA_MODEL"]:
        f.write(f"{key}={os.environ.get(key, '')}\n")

print("API keys set")

Enter HuggingFace token: ··········
API keys set


In [6]:
import yaml

with open("configs/training_config.yaml", "r") as f:
    config = yaml.safe_load(f)

config["model"]["base_model"] = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
config["model"]["max_length"]  = 512

with open("configs/training_config.yaml", "w") as f:
    yaml.dump(config, f)

with open("configs/lora_config.yaml", "r") as f:
    lora_config = yaml.safe_load(f)

lora_config["lora"]["target_modules"] = [
    "q_proj", "k_proj", "v_proj", "o_proj"
]
lora_config["quantization"]["load_in_4bit"] = False

with open("configs/lora_config.yaml", "w") as f:
    yaml.dump(lora_config, f)

print(f"Config updated for TinyLlama")
print(f"Model: {config['model']['base_model']}")

Config updated for TinyLlama
Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0


In [7]:
import shutil
import os

cache_dir   = os.path.expanduser("~/.cache/huggingface/datasets")
spider_cache = os.path.join(cache_dir, "xlangai___spider")

if os.path.exists(spider_cache):
    shutil.rmtree(spider_cache)
    print(f"Cleared cache: {spider_cache}")
else:
    print("No cache to clear")

No cache to clear


In [8]:
import sys
sys.path.insert(0, '.')

from src.data.dataset_loader import SpiderDatasetLoader
from src.data.prompt_formatter import SQLPromptFormatter

loader      = SpiderDatasetLoader()
train, eval = loader.load()

print(f"Train examples: {len(train)}")
print(f"Eval examples:  {len(eval)}")

formatter = SQLPromptFormatter()
print("\n=== EXAMPLE TRAINING PROMPT ===")
print(formatter.format_training(train[0]))

stats = loader.get_statistics(train)
print("\n=== DATASET STATISTICS ===")
for k, v in stats.items():
    print(f"  {k}: {v}")

2026-06-18 14:56:48 | INFO | src.data.dataset_loader | Initializing Spider dataset loader
2026-06-18 14:56:48 | INFO | src.data.dataset_loader | Loading Spider dataset from HuggingFace


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/5.51k [00:00<?, ?B/s]

spider/train-00000-of-00001.parquet:   0%|          | 0.00/831k [00:00<?, ?B/s]

spider/validation-00000-of-00001.parquet:   0%|          | 0.00/126k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]

2026-06-18 14:56:56 | INFO | src.data.dataset_loader | Dataset loaded
Train examples: 1000
Eval examples:  200

=== EXAMPLE TRAINING PROMPT ===
### Task
Convert the following natural language question to a SQL query.

### Database Schema
Database: department_management

### Question
How many heads of the departments are older than 56 ?

### SQL
SELECT count(*) FROM head WHERE age  >  56

=== DATASET STATISTICS ===
  total: 1000
  difficulties: {'unknown': 1000}
  avg_sql_length: 14.612
  avg_question_length: 12.208
  unique_databases: 19
  sql_keywords: {'SELECT': 1000, 'FROM': 1000, 'WHERE': 475, 'JOIN': 332, 'GROUP BY': 265, 'ORDER BY': 244, 'HAVING': 74, 'LIMIT': 181, 'UNION': 6, 'INTERSECT': 29}


## Load Base Model with LoRA

Loading TinyLlama-1.1B with LoRA adapters.

LoRA math recap:
- Original weights: FROZEN
- new_output = W·x + B·A·x × (alpha/r)
- Only A and B matrices are trained
- Reduces trainable params from 1.1B to ~5M

In [10]:
import subprocess

print("Upgrading torchao...")
subprocess.run(["pip", "install", "--upgrade", "torchao", "-q"])
print("✓ torchao upgraded")


Upgrading torchao...
✓ torchao upgraded


In [11]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import get_peft_model, LoraConfig, TaskType

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model in float16...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("Applying LoRA adapters...")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_config)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal params:     {total:,}")
print(f"Trainable params: {trainable:,} ({100*trainable/total:.2f}%)")
print("\nModel ready!")

Loading tokenizer...
Loading model in float16...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Applying LoRA adapters...

Total params:     1,104,553,984
Trainable params: 4,505,600 (0.41%)

Model ready!


In [12]:
formatter     = SQLPromptFormatter()
test_examples = eval[:3]

print("BASE MODEL PREDICTIONS (before fine-tuning)\n")
print("="*60)

for i, example in enumerate(test_examples):
    prompt = formatter.format_inference(example)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    sql = formatter.extract_sql(generated)

    print(f"Q{i+1}: {example.question}")
    print(f"Expected:  {example.sql}")
    print(f"Generated: {sql}")
    print()

BASE MODEL PREDICTIONS (before fine-tuning)



[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q1: How many singers do we have?
Expected:  SELECT count(*) FROM singer
Generated: ```
SELECT COUNT(*) FROM concert_singer;
```



[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q2: What is the total number of singers?
Expected:  SELECT count(*) FROM singer
Generated: ```
SELECT COUNT(*) FROM concert_singer;
```

Q3: Show name, country, age for all singers ordered by age from the oldest to the youngest.
Expected:  SELECT name ,  country ,  age FROM singer ORDER BY age DESC
Generated: ```
SELECT singer.name, singer.country, singer.age
FROM singer
ORDER BY singer.age DESC;
```



In [13]:
from src.data.preprocessor import DataPreprocessor

print("Preprocessing training data...")
preprocessor  = DataPreprocessor(tokenizer)
train_dataset = preprocessor.prepare(train)
eval_dataset  = preprocessor.prepare(eval)

print(f"Train dataset: {len(train_dataset)} examples")
print(f"Eval dataset:  {len(eval_dataset)} examples")
print(f"Features: {train_dataset.features}")

Preprocessing training data...
2026-06-18 15:00:33 | INFO | src.data.preprocessor | Preprocessing 1000 examples
2026-06-18 15:00:35 | INFO | src.data.preprocessor | Preprocessing complete
2026-06-18 15:00:36 | INFO | src.data.preprocessor | Preprocessing 200 examples
2026-06-18 15:00:36 | INFO | src.data.preprocessor | Preprocessing complete
Train dataset: 1000 examples
Eval dataset:  200 examples
Features: {'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8')), 'labels': List(Value('int64'))}


## Fine-tune with LoRA

Training configuration:
- 3 epochs over 1000 training examples
- Effective batch size = 4 × 4 = 16
- Learning rate 2e-4 with 100 warmup steps
- Loss computed ONLY on SQL tokens
- Progress logged every 10 steps

Expected training time: 20-40 minutes on T4 GPU

In [17]:
import os
from transformers import TrainingArguments, Trainer
from src.training.callbacks import ProgressCallback

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

training_args = TrainingArguments(
    output_dir="outputs/sql-finetune",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=100,
    logging_steps=10,
    eval_steps=100,
    fp16=True,
    dataloader_num_workers=0,
    remove_unused_columns=False,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    callbacks=[ProgressCallback()],
)

total_steps = (
    len(train_dataset) //
    (training_args.per_device_train_batch_size *
     training_args.gradient_accumulation_steps) *
    int(training_args.num_train_epochs)
)
print(f"Starting training... Total steps: {total_steps}")
trainer.train()
print("\nTraining complete!")

Starting training... Total steps: 186


Step,Training Loss,Validation Loss


  Step 10/189 (5%) | loss=16.9312 | lr=1.80e-05
  Step 20/189 (11%) | loss=12.9220 | lr=3.80e-05
  Step 30/189 (16%) | loss=4.9419 | lr=5.80e-05
  Step 40/189 (21%) | loss=0.4865 | lr=7.80e-05
  Step 50/189 (26%) | loss=0.0957 | lr=9.80e-05
  Step 60/189 (32%) | loss=0.0754 | lr=1.18e-04
  Step 70/189 (37%) | loss=0.0625 | lr=1.38e-04


Step,Training Loss,Validation Loss
100,0.041810,0.079890
189,0.023004,0.085706


  Step 80/189 (42%) | loss=0.0606 | lr=1.58e-04
  Step 90/189 (48%) | loss=0.0447 | lr=1.78e-04
  Step 100/189 (53%) | loss=0.0418 | lr=1.98e-04
  Step 100/189 (53%) | loss=? | lr=?
  Step 110/189 (58%) | loss=0.0346 | lr=1.80e-04
  Step 120/189 (63%) | loss=0.0348 | lr=1.57e-04
  Step 130/189 (69%) | loss=0.0344 | lr=1.35e-04
  Step 140/189 (74%) | loss=0.0287 | lr=1.12e-04
  Step 150/189 (79%) | loss=0.0274 | lr=8.99e-05
  Step 160/189 (85%) | loss=0.0263 | lr=6.74e-05
  Step 170/189 (90%) | loss=0.0245 | lr=4.49e-05
  Step 180/189 (95%) | loss=0.0230 | lr=2.25e-05
  Step 189/189 (100%) | loss=? | lr=?
  Step 189/189 (100%) | loss=1.9004 | lr=?

Training complete!
